# Init

In [1]:
import os, pickle
from random import choice
from collections import deque

import cv2
import numpy as np
import matplotlib.pyplot as plt

from aspire.env_config import env_var

from homog_utils import posn_from_xform

In [2]:
class TColor:
    """ Terminal Colors """
    # Source: https://stackoverflow.com/a/287944
    HEADER    = '\033[95m'
    OKBLUE    = '\033[94m'
    OKCYAN    = '\033[96m'
    OKGREEN   = '\033[92m'
    WARNING   = '\033[93m'
    FAIL      = '\033[91m'
    ENDC      = '\033[0m'
    BOLD      = '\033[1m'
    UNDERLINE = '\033[4m'

In [3]:
_DATA_DRIVE = "DATA_TANK"

tests = [
    "KC-KP",
    "SC-KP",
    "KC-SP",
    "SC-SP",
]

In [4]:
########## SETUP ###################################################################################
_JSON_PATH  = "data/allData.txt"
_DATA_DRIVE = "DATA_TANK"
# _PLOT_DIR   = "/media/james/FILEPILE/EROM/data/plots/"
_PLOT_DIR   = "data/plots/"

tests = [
    "KC-KP",
    "SC-KP",
    "KC-SP",
    "SC-SP",
]

longTestNames = [
    "Known Class & Known Pose", 
    "Sensed Class & Known Pose", 
    "Known Class & Sensed Pose", 
    "Sensed Class & Sensed Pose", 
]

# paths = [ f"/media/james/{_DATA_DRIVE}/2025-08_{test}" for test in tests ]
datasets = [
    [ f"/media/james/{_DATA_DRIVE}/2025-08B_{test}" for test in tests ],
    [ f"/media/james/{_DATA_DRIVE}/RWB_2025-09_{test}" for test in tests ],
]

dataLabels = ["RGB", "RBW",]
datNamLong = {
    "RGB": "Red-Green-Blue", 
    "RBW": "Red-Black-White",
}

fNames = [ f"{_PLOT_DIR}{test}" for test in tests  ]

plotExt = ".pdf"

pkls = deque()

for iii, paths in enumerate( datasets ):
    setNam = dataLabels[iii]
    suffix = "_" + setNam
    skip   = False

    for ii, test in enumerate( tests ):
        ##### Init ################################################################
        path     = paths[ii]
        fName    = fNames[ii]
        longTNam = longTestNames[ii]

        ##### Load ################################################################
        try:
            pkls.extend( [os.path.join( path, item ) for item in os.listdir( path ) if ".pkl" in f"{item}".lower()] )
        except FileNotFoundError as e:
            print( f"\n404, SKIP THIS TEST: {e}\n" )
            skip = True
            continue

# "Ground Truth" Annotator

In [5]:
from State import OCV_State_Tracker
ocvTracker = OCV_State_Tracker()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D INFO] Resetting default logger to print to terminal.


# Load File(s)

In [6]:
_EXP_INDEX = 0 # 130
_SAFE_POSN = np.array( [-0.25199, -0.26192, 0.47106,] ) 

In [7]:
print( f"Found {len(pkls)} files!" ) 
pklFile = pkls[ _EXP_INDEX ]

data = list()
try:
    with open( pklFile, 'rb' ) as f:
        data = pickle.load( f )
        print( f"LOAD SUCCESS: {pklFile}" )
except EOFError as e:
    print( f"LOAD ERROR: {e}" )

Found 162 files!
LOAD SUCCESS: /media/james/DATA_TANK/2025-08B_KC-KP/EROM-Memories_08-25-2025_11-18-58.pkl


In [8]:
from magpie_control.ur5 import _CAMERA_XFORM

def camPose_from_effPose( effPose : np.ndarray ):
    """ Get the camera pose given the effector pose """
    return np.dot( effPose, _CAMERA_XFORM )

### 1. Object Search ###
shotPoses = list()

In [9]:
for i, datum in enumerate( data ):
    dtmMsg = datum['msg']
    dtmDat = datum['data']

    print( f"\n{TColor.OKBLUE}MESSAGE: {dtmMsg}{TColor.ENDC}\n" ) 

    ##### General State ###################################

    if "RobotState" in dtmMsg:
        pose_t = np.array( dtmDat['pose'] )
        posn_t = posn_from_xform( pose_t )
        if np.linalg.norm( posn_t - _SAFE_POSN ) > env_var("_ACCEPT_POSN_ERR"):
            print( f"Moved to {posn_t}" )
            shotPoses.append( pose_t )  

    
    ##### Phase 1: Perception #############################
    
    if "BGN: Phase 1" in dtmMsg:
        # ASSUMPTION: PHASE 1 MESSAGE SENT ONLY ONCE PER STEP, See `p1pp2`
        res = ocvTracker.process_ray_obs()
        print( f"Last Scene: {res}" )
        ocvTracker.new_scene()
    
    if dtmMsg == "ObsMeta":
        # inpt = dtmDat['input']
        # hits = dtmDat['hits']
        # snap = inpt[ choice( list( inpt.keys() ) ) ]
        # # print( list( snap.keys() ) )
        # imag = snap['image']
        # dpth = snap['depth']
        # fig  = plt.figure()
        # plt.imshow( imag )
        # plt.close( fig )
        # fig = plt.figure()
        # plt.imshow( dpth )
        # plt.close( fig )
        # ocvTracker.TEST_FUNC( 'redBlock', imag )
        # res = ocvTracker.find_block_mask( 'redBlock', imag, dpth )
        ocvTracker.process_observation_data( dtmDat, ['redBlock','grnBlock','bluBlock',], camPose_from_effPose( shotPoses[-1] ) )
        
res = ocvTracker.process_ray_obs()
print( f"Last Scene: {res}" )
        


MESSAGE: RobotState


MESSAGE: RobotState


MESSAGE: Task Start


MESSAGE: Performance


MESSAGE: Performance


MESSAGE: BGN: Phase 1

Last Scene: None

MESSAGE: Performance


MESSAGE: RobotState


MESSAGE: RobotState

Moved to [-0.554  0.169  0.258]

MESSAGE: Observation BEGIN


MESSAGE: Annotation


MESSAGE: ObsMeta

['query', 'abbrv', 'image', 'depth', 't']
0.0
Block mask of 20452 points!
Block mask is 0.19131192564964294 away!
Span is [173, 167]
Arc is [np.float64(0.2432310238126542), np.float64(0.19127717771075356)]
Scale is [1.169 0.918] * 0.04
Block mask of 16 points!
MASK FOUND for redBlock!
0.0
Block mask of 13560 points!
Block mask is 0.2508141100406647 away!
Span is [136, 141]
Arc is [np.float64(0.191210515829601), np.float64(0.16149749734860028)]
Scale is [1.203 1.015] * 0.04
Block mask of 529 points!
Block mask of 500 points!
Block mask of 456 points!
MASK FOUND for grnBlock!
0.0
Block mask of 7999 points!
Block mask of 7169 points!
Block mask is 0.3126554489135742 away!
